# EXIT retrieval: public Kaggle Dataset checkpoints
Prepare once in /tmp; upload directly to public Datasets. Checkpoint + upload every 120 minutes, plus stage completion and graceful stop. No large files are copied into /kaggle/working. Set your Kaggle username and enable Internet before running. See retrieval/README.md.

In [ ]:
from pathlib import Path
import os
REPO = Path('/kaggle/working/EXIT')  # adjust checkout path
os.chdir(REPO)

In [ ]:
%pip install -r requirements-retrieval.txt

## Configure
This cell writes a notebook-local config copy, without changing the repo template. Existing state in /tmp/retrieval-state is reused. Use a new Dataset prefix when changing PQ settings. All generated Datasets are public.

In [ ]:
import json
config = json.loads(Path('configs/retrieval_kaggle.json').read_text())
config['owner'] = 'YOUR_USERNAME'  # REQUIRED: your actual Kaggle username
config['dataset_prefix'] = 'exit-wikipedia-retrieval'
config['checkpoint_minutes'] = 120
config['time_budget_minutes'] = 600
# Optional attached input sources:
# config['shards'] = '/kaggle/input/embedding-source/passages_*'
# config['tsv'] = '/kaggle/input/wiki-corpus/psgs_w100.tsv'
# Optional checkpoint/corpus attached via Add Input:
# config['checkpoint_input'] = '/kaggle/input/exit-wikipedia-retrieval-checkpoint'
# config['corpus_input'] = '/kaggle/input/exit-wikipedia-retrieval-corpus'
assert config['owner'] != 'YOUR_USERNAME', 'Set your Kaggle username first'
CONFIG = '/kaggle/working/retrieval-session.json'
Path(CONFIG).write_text(json.dumps(config, indent=2))

## Authentication
Create and enable a Kaggle Secret named KAGGLE_API_TOKEN. It is passed only through the environment; it is not included in snapshots. For a legacy key, use the commented alternative.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['KAGGLE_API_TOKEN'] = secrets.get_secret('KAGGLE_API_TOKEN')
# Legacy alternative (replace the token line above):
# os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')

In [ ]:
!python -m retrieval.prepare preflight --work /tmp/retrieval-state

## Restore only in a new session
Leave commented for a first run or when continuing with intact local state. If enabled, restore downloads the latest checkpoint and exact referenced corpus version (or reads the configured attached Inputs). Conflicting local files are not overwritten.

In [ ]:
# !python -m retrieval.prepare restore --config "{CONFIG}"

## Check status
Read-only. Shows completed/current steps, counts, the next step and confirmed remote Dataset versions. Local progress is separate from remotely saved progress.

In [ ]:
!python -m retrieval.prepare status --config "{CONFIG}"

## Resume automatically
Runs missing stages until completion or the processing time budget. Upload failures stop processing and retain snapshots. Leave time for uploads before the Kaggle session limit. Train PQ and individual extraction/hash operations cannot be paused mid-operation.

In [ ]:
!python -m retrieval.prepare resume --config "{CONFIG}"

In [ ]:
!python -m retrieval.prepare status --config "{CONFIG}"

## Optional: save durable state immediately
Run only after the active resume command has stopped. This uploads the last saved state, not another process's live in-memory index. Automatic resume already saves at budget stop and stage completion.

In [ ]:
# !python -m retrieval.prepare checkpoint --config "{CONFIG}"

## Next session / inference
New preparation session: load config and Secrets, enable restore, then run status and resume. Finished: attach the public Dataset <owner>/<prefix>-artifact to the inference notebook. Its four artifact files can be used directly; pack/restore packages are unnecessary for the new workflow. Raw input downloads are not checkpointed remotely and may need downloading again.